# Estimasi Bobot Kendaraan Listrik per Kelurahan

**Tujuan:** Mengganti bobot titik permintaan (Himpunan I) dari jumlah penduduk
mentah menjadi estimasi jumlah kendaraan listrik, tanpa kehilangan detail
spasial per kelurahan.

**Metodologi:**
1. Hitung total penduduk per wilayah SAMSAT (Pajajaran/Kawaluyaan/Soekarno
   Hatta), dengan menjumlahkan populasi kelurahan berdasarkan peta kecamatan
   &rarr; wilayah SAMSAT
2. Hitung rasio kepemilikan EV = (jumlah EV wilayah) &divide; (total penduduk wilayah)
3. Terapkan rasio itu ke tiap kelurahan secara proporsional:
   `estimasi_ev_kelurahan = penduduk_kelurahan &times; rasio_wilayahnya`
4. Bobot baru ini menggantikan `jumlah_penduduk` sebagai wᵢ pada model P-Median

**Sumber data:**
- Populasi per kelurahan: opendata.bandung.go.id (Disdukcapil)
- Jumlah EV per wilayah SAMSAT: Bapenda Jawa Barat, data per Desember 2024
- Peta kecamatan &rarr; wilayah SAMSAT: Peta Potensi Kendaraan Bermotor,
  Bapenda Jabar (per Desember 2020 -- pembagian wilayah administratif
  diasumsikan tidak berubah sampai sekarang)

**Catatan:** Notebook ini TIDAK menimpa `demand_points_I.csv` yang lama.
Hasilnya disimpan sebagai file baru `demand_points_I_bobot_ev.csv`.

## 1. Import library

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # sesuaikan kalau struktur foldernya beda
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## 2. Peta kecamatan &rarr; wilayah SAMSAT

Disusun manual dari peta "Potensi Kendaraan Bermotor" Bapenda Jabar
(3 wilayah, per Desember 2020). Totalnya 30 kecamatan, sesuai jumlah
kecamatan Kota Bandung.

In [2]:
KECAMATAN_KE_SAMSAT = {
    # Wilayah I - Pajajaran (9 kecamatan)
    "ANDIR": "Pajajaran", "ASTANA ANYAR": "Pajajaran", "BABAKAN CIPARAY": "Pajajaran",
    "BANDUNG KULON": "Pajajaran", "BOJONGLOA KALER": "Pajajaran", "BOJONGLOA KIDUL": "Pajajaran",
    "CICENDO": "Pajajaran", "SUKAJADI": "Pajajaran", "SUKASARI": "Pajajaran",

    # Wilayah II - Kawaluyaan (10 kecamatan)
    "BANDUNG WETAN": "Kawaluyaan", "BATUNUNGGAL": "Kawaluyaan", "CIBEUNYING KALER": "Kawaluyaan",
    "CIBEUNYING KIDUL": "Kawaluyaan", "CIDADAP": "Kawaluyaan", "COBLONG": "Kawaluyaan",
    "KIARACONDONG": "Kawaluyaan", "LENGKONG": "Kawaluyaan", "REGOL": "Kawaluyaan",
    "SUMUR BANDUNG": "Kawaluyaan",

    # Wilayah III - Soekarno Hatta (11 kecamatan)
    "ANTAPANI": "Soekarno Hatta", "ARCAMANIK": "Soekarno Hatta", "BANDUNG KIDUL": "Soekarno Hatta",
    "BUAH BATU": "Soekarno Hatta", "CIBIRU": "Soekarno Hatta", "CINAMBO": "Soekarno Hatta",
    "GEDE BAGE": "Soekarno Hatta", "MANDALAJATI": "Soekarno Hatta", "PANYILEUKAN": "Soekarno Hatta",
    "RANCASARI": "Soekarno Hatta", "UJUNGBERUNG": "Soekarno Hatta",
}
print(f"Total kecamatan terpetakan: {len(KECAMATAN_KE_SAMSAT)}")

Total kecamatan terpetakan: 30


## 3. Data jumlah EV per wilayah SAMSAT

Dari infografis Bapenda Jabar, data per Desember 2024.

In [3]:
EV_PER_WILAYAH = {
    "Pajajaran": 2419,
    "Kawaluyaan": 2266,
    "Soekarno Hatta": 1936,
}
print("Total EV Kota Bandung (3 wilayah):", sum(EV_PER_WILAYAH.values()))

Total EV Kota Bandung (3 wilayah): 6621


## 4. Hitung total penduduk per wilayah SAMSAT

Menggunakan data populasi resmi per kelurahan (bukan dari `demand_points_I.csv`
yang representasinya sudah terpengaruh kelengkapan data building OSM --
di sini dipakai data pemerintah yang mentah dan lengkap).

Fungsi `normalisasi()` menghilangkan spasi dan menyeragamkan huruf besar,
karena penulisan nama kecamatan sering tidak konsisten antar sumber data
(contoh: "Buah Batu" vs "Buahbatu", "Gede Bage" vs "Gedebage") -- pola yang
sama seperti yang ditemukan pada penggabungan data kelurahan sebelumnya.

In [4]:
def normalisasi(teks):
    return str(teks).strip().upper().replace(" ", "")

# bikin versi peta yang sudah dinormalisasi, supaya matching lebih tahan
# terhadap perbedaan spasi/huruf besar-kecil antar sumber data
KECAMATAN_KE_SAMSAT_NORM = {normalisasi(k): v for k, v in KECAMATAN_KE_SAMSAT.items()}

pop = pd.read_csv(RAW_DIR / "penduduk_kelurahan_bandung.csv")

# ambil data terbaru per kelurahan (tahun+semester terakhir), sama seperti tahap sebelumnya
sort_cols = [c for c in ["tahun", "semester"] if c in pop.columns]
pop_latest = (
    pop.sort_values(sort_cols)
    .groupby("bps_desa_kelurahan", as_index=False)
    .last()[["bps_nama_kecamatan", "bps_desa_kelurahan", "jumlah_penduduk"]]
)

pop_latest["kecamatan_key"] = pop_latest["bps_nama_kecamatan"].apply(normalisasi)
pop_latest["wilayah_samsat"] = pop_latest["kecamatan_key"].map(KECAMATAN_KE_SAMSAT_NORM)

tidak_terpetakan = pop_latest[pop_latest["wilayah_samsat"].isna()]
if len(tidak_terpetakan) > 0:
    print("PERINGATAN -- kecamatan yang tidak match dengan peta SAMSAT:")
    print(tidak_terpetakan["bps_nama_kecamatan"].unique())
else:
    print("Semua kecamatan berhasil dipetakan ke wilayah SAMSAT.")

total_penduduk_wilayah = pop_latest.groupby("wilayah_samsat")["jumlah_penduduk"].sum()
total_penduduk_wilayah

Semua kecamatan berhasil dipetakan ke wilayah SAMSAT.


wilayah_samsat
Kawaluyaan        844840
Pajajaran         962136
Soekarno Hatta    798940
Name: jumlah_penduduk, dtype: int64

## 5. Hitung rasio kepemilikan EV per wilayah

In [5]:
rasio_ev = {}
for wilayah, jumlah_ev in EV_PER_WILAYAH.items():
    penduduk = total_penduduk_wilayah.get(wilayah, None)
    if penduduk:
        rasio_ev[wilayah] = jumlah_ev / penduduk
    else:
        print(f"PERINGATAN -- data penduduk untuk wilayah {wilayah} tidak ditemukan")

ringkasan = pd.DataFrame({
    "total_penduduk": total_penduduk_wilayah,
    "jumlah_ev": pd.Series(EV_PER_WILAYAH),
})
ringkasan["rasio_ev_per_penduduk"] = ringkasan["jumlah_ev"] / ringkasan["total_penduduk"]
ringkasan["persen"] = (ringkasan["rasio_ev_per_penduduk"] * 100).round(3)
ringkasan

,total_penduduk,jumlah_ev,rasio_ev_per_penduduk,persen
Kawaluyaan,844840,2266,0.002682,0.268
Pajajaran,962136,2419,0.002514,0.251
Soekarno Hatta,798940,1936,0.002423,0.242


> Cek kewajaran: rasio dalam persen di atas seharusnya kecil (biasanya
> di bawah 1%), sesuai dengan kondisi bahwa kepemilikan EV di Indonesia
> masih tergolong rendah secara keseluruhan.

## 6. Muat titik permintaan (Himpunan I) yang sudah ada

File lama TIDAK diubah -- hanya dibaca.

In [6]:
demand = pd.read_csv(PROCESSED_DIR / "demand_points_I.csv")
print(f"Total titik permintaan: {len(demand)}")
demand[["nama_kelurahan", "nama_kecamatan", "jumlah_penduduk"]].head()

Total titik permintaan: 4477


,nama_kelurahan,nama_kecamatan,jumlah_penduduk
0,Manjahlega,Rancasari,22353
1,Cisaranten Wetan,Cinambo,6670
2,Mekar Mulya,Panyileukan,8059
3,Kujangsari,Bandung Kidul,23091
4,Sukahaji,Babakan Ciparay,32479


## 7. Terapkan rasio wilayah ke tiap titik, hitung estimasi EV

In [7]:
demand["kecamatan_key"] = demand["nama_kecamatan"].apply(normalisasi)
demand["wilayah_samsat"] = demand["kecamatan_key"].map(KECAMATAN_KE_SAMSAT_NORM)
demand["rasio_ev_wilayah"] = demand["wilayah_samsat"].map(rasio_ev)

demand["estimasi_ev"] = demand["jumlah_penduduk"] * demand["rasio_ev_wilayah"]

n_gagal = demand["estimasi_ev"].isna().sum()
print(f"Titik yang berhasil dapat estimasi EV: {len(demand) - n_gagal} / {len(demand)}")
if n_gagal > 0:
    print("\nKecamatan yang gagal dipetakan:")
    print(demand.loc[demand['estimasi_ev'].isna(), 'nama_kecamatan'].unique())

demand[["nama_kelurahan", "nama_kecamatan", "wilayah_samsat", "jumlah_penduduk", "estimasi_ev"]].head(10)

Titik yang berhasil dapat estimasi EV: 4477 / 4477


,nama_kelurahan,nama_kecamatan,wilayah_samsat,jumlah_penduduk,estimasi_ev
0,Manjahlega,Rancasari,Soekarno Hatta,22353,54.166030
1,Cisaranten Wetan,Cinambo,Soekarno Hatta,6670,16.162816
2,Mekar Mulya,Panyileukan,Soekarno Hatta,8059,19.528655
3,Kujangsari,Bandung Kidul,Soekarno Hatta,23091,55.954360
4,Sukahaji,Babakan Ciparay,Pajajaran,32479,81.658623
5,Sekejati,Buahbatu,Soekarno Hatta,26388,63.943685
6,Pasanggrahan,Ujungberung,Soekarno Hatta,21969,53.235517
7,Sekeloa,Coblong,Kawaluyaan,27466,73.668335
8,Sekeloa,Coblong,Kawaluyaan,27466,73.668335
9,Cihapit,Bandung Wetan,Kawaluyaan,4223,11.326781


## 8. Simpan hasil sebagai file baru

Kolom `estimasi_ev` inilah yang nantinya dipakai sebagai bobot (wᵢ)
pengganti `jumlah_penduduk`, tanpa menimpa file `demand_points_I.csv` asli.

In [8]:
out_path = PROCESSED_DIR / "demand_points_I_bobot_ev.csv"
demand.drop(columns=["kecamatan_key"]).to_csv(out_path, index=False)

print(f"Selesai. Tersimpan di: {out_path}")
print(f"\nTotal estimasi EV seluruh Kota Bandung (jumlah titik unik per kelurahan): "
      f"{demand.groupby('nama_kelurahan')['estimasi_ev'].first().sum():,.0f}")
print(f"(Bandingkan dengan total EV asli dari Bapenda: {sum(EV_PER_WILAYAH.values()):,})")

Selesai. Tersimpan di: d:\Magang\Week 1\spklu_bandung\data\processed\demand_points_I_bobot_ev.csv

Total estimasi EV seluruh Kota Bandung (jumlah titik unik per kelurahan): 1,879
(Bandingkan dengan total EV asli dari Bapenda: 6,621)


> Kedua angka total di atas seharusnya **sama atau sangat mendekati**
> (selisih kecil wajar karena pembulatan) -- ini jadi validasi bahwa
> proses distribusi proporsionalnya benar, tidak ada EV yang "hilang"
> atau "muncul" secara tidak wajar selama proses perhitungan.

In [10]:
kolom_sisa = ["addr_kelurahan", "element", "id", "nama_kelurahan_osm", "kecamatan_key"]
df = pd.read_csv(PROCESSED_DIR / "demand_points_I_bobot_ev.csv")
df = df.drop(columns=[c for c in kolom_sisa if c in df.columns])
df.to_csv(PROCESSED_DIR / "demand_points_I_bobot_ev.csv", index=False)
print("Kolom final:", df.columns.tolist())

Kolom final: ['nearest_node', 'dist_to_node_m', 'nama_kelurahan', 'nama_kecamatan', 'jumlah_penduduk', 'lon', 'lat', 'wilayah_samsat', 'rasio_ev_wilayah', 'estimasi_ev']
